[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github//MissingSemesterCUET/ML/blob/main/lectures/02.02/notebook2.ipynb)

# Class 3: NumPy from Scratch (Part II)

In [2]:
import numpy as np

In [5]:
class ndarray:
    def __init__(self, data):
        self.data = data
        self.shape = (len(data),)
    
    def __repr__(self):
        return f"array({self.data})"

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, index):
        return self.data[index]

    def __setitem__(self, index, value):
        self.data[index] = value
    

def array(data):
    return ndarray(data)

# v3: Element-wise Operations

The important thing to realize is that when we write
```py
c = a + b
```

Python actually calls
```py
c = a.__add__(b)
```

Similarly,
```py
a - b   --> a.__sub__(b)
a * b   --> a.__mul__(b)
a / b   --> a.__truediv__(b)
```

So we'll implement those methods directly.

In [6]:
class ndarray(ndarray):
    def __add__(self, other):
        result = []

        if isinstance(other, ndarray):
            if self.shape != other.shape:
                raise ValueError("Shapes must match.")

            for a, b in zip(self.data, other.data):
                result.append(a + b)

        else:  # scalar
            for a in self.data:
                result.append(a + other)

        return ndarray(result)

    def __sub__(self, other):
        result = []

        if isinstance(other, ndarray):
            if self.shape != other.shape:
                raise ValueError("Shapes must match.")

            for a, b in zip(self.data, other.data):
                result.append(a - b)

        else:
            for a in self.data:
                result.append(a - other)

        return ndarray(result)

    def __mul__(self, other):
        result = []

        if isinstance(other, ndarray):
            if self.shape != other.shape:
                raise ValueError("Shapes must match.")

            for a, b in zip(self.data, other.data):
                result.append(a * b)

        else:
            for a in self.data:
                result.append(a * other)

        return ndarray(result)

    def __truediv__(self, other):
        result = []

        if isinstance(other, ndarray):
            if self.shape != other.shape:
                raise ValueError("Shapes must match.")

            for a, b in zip(self.data, other.data):
                result.append(a / b)

        else:
            for a in self.data:
                result.append(a / other)

        return ndarray(result)


def array(data):
    return ndarray(data)

In [7]:
a = array([1, 2, 3])
b = array([4, 5, 6])

print(a + b)
# array([5, 7, 9])

print(a - b)
# array([-3, -3, -3])

print(a * b)
# array([4, 10, 18])

print(b / a)
# array([4.0, 2.5, 2.0])

print(a + 10)
# array([11, 12, 13])

print(a * 5)
# array([5, 10, 15])

array([5, 7, 9])
array([-3, -3, -3])
array([4, 10, 18])
array([4.0, 2.5, 2.0])
array([11, 12, 13])
array([5, 10, 15])


**A small observation**

You probably noticed that __add__, __sub__, __mul__, and __truediv__ are almost identical. Real libraries avoid this repetition by having all four call a common internal helper. For learning purposes, though, writing them out separately makes it much easier to see exactly what each operator is doing before we start refactoring.

# v4: Reduction Methods (Exercise)

In [8]:
a = np.array([1, 2, 3, 4])

print(a.sum())     # 10
print(a.mean())    # 2.5
print(a.min())     # 1
print(a.max())     # 4

10
2.5
1
4


Home work:
- `prod()`
- `std()`
- `var()`
- `argmin()`
- `argmax()`

**What you just learned**

Until now, every operation returned another array.

a + b --> array([...])

A reduction is different.

array([5, 1, 8, 2]) --->reduce(sum)---> 16

Instead of producing another array, it produces a single number.

# v5: 2D (Part I)

Until now, our array has only supported 1D data.

But NumPy is designed for N-dimensional arrays.

Let's first support 2D arrays.

Currently we have
```py
self.shape = (len(data),)
```

Instead, detect whether data is 1D or 2D.

In [9]:
class ndarray(ndarray):
    def __init__(self, data):
        self.data = data
        if isinstance(data[0], list):
            row = len(data)
            col = len(data[0])
            self.shape = (row, col)
        else:
            self.shape = (len(data),)

a = array([[1,2,3],[4,5,6]])
a.shape

(2, 3)

**Validate the input**

What if someone writes

```py
array([
    [1,2],
    [3,4,5]
])
```

Is this a valid matrix?

In [10]:
np.array([[1,2,9],[3,4,5,],[3,4,5,0]])

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (3,) + inhomogeneous part.

Every row must have the same length.

Add this check:

In [11]:
class ndarray(ndarray):
    def __init__(self, data):
        self.data = data
        if isinstance(data[0], list):
            len_row = len(data)
            len_col = len(data[0])
            for row in data:
                if len(row) != len_col:
                    raise ValueError(f"setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was ({row},) + inhomogeneous part.")

            self.shape = (len_row, len_col)
        else:
            self.shape = (len(data),)

a = array([[1,2,3],[4,5]])

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was ([4, 5],) + inhomogeneous part.

Now we have an issue with indexing:

In [12]:
a = np.array([
    [1,2,3],
    [4,5,6]
])
a[0]

array([1, 2, 3])

But in our module:

In [13]:
a = array([
    [1,2,3],
    [4,5,6]
])
a[0]

[1, 2, 3]

It doesn't return a ndarray. It returns a regular python list

So, let's fix it

In [14]:
class ndarray(ndarray):
    def __getitem__(self, index):
        value = self.data[index]
        if isinstance(value, list):
            return ndarray(value)
        return value

# now let's try it, again
a = array([
    [1,2,3],
    [4,5,6]
])
a[0]

array([1, 2, 3])

But there's a problem...

Our addition code assumes the data is flat.

For example,

In [15]:
a = array([
    [1,2],
    [3,4]
])

b = array([
    [5,6],
    [7,8]
])

a + b

array([[1, 2, 5, 6], [3, 4, 7, 8]])

Our current loop does

```py
for a, b in zip(self.data, other.data):
    result.append(a + b)
```

But here `a` is actually `[1,2]` and `b` is `[5,6]`

Python list addition means concatenation, so you'd get:

[1,2] + [5,6] = [1,2,5,6]

instead of

[1,2] + [5,6] = [6,8]

So the next stage is where we'll make our arithmetic truly N-dimensional by teaching it to recursively apply operations to nested lists. That will let the same +, -, *, and / code work for both 1D and 2D arrays, bringing our implementation much closer to how NumPy thinks about arrays.

# v6: 2D (Part II)

This is where our implementation starts to resemble the real design of NumPy.

Instead of writing separate code for 1D arrays and 2D arrays, we'll think recursively.

**The idea**

Consider these two arrays:

```py
a = [
    [1, 2],
    [3, 4]
]

b = [
    [5, 6],
    [7, 8]
]
```

What is a + b?

Don't think about the whole matrix at once. Think recursively.
```py
[
    [1,2] + [5,6],
    [3,4] + [7,8]
]
```

Now break it down again.
```py
[
    [
        1+5,
        2+6
    ],
    [
        3+7,
        4+8
    ]
]
```

Eventually you reach numbers, and numbers know how to add themselves.

This suggests a recursive algorithm.

First, let's add a recursive helper

Add this method to your class.

In [16]:
class ndarray(ndarray):
    def _apply_op(self, left, right, op):
        if not isinstance(left, list):
            return op(left, right)

        result = []

        for a, b in zip(left, right):
            result.append(self._apply_op(a, b, op))

        return result

Notice that this method doesn't know whether it's working on a vector, matrix, or higher-dimensional tensor.

It simply asks:

"Am I looking at numbers?"

Yes --> perform the operation.
No --> go one level deeper.

Next, we rewrite __add__

Now __add__ becomes much shorter.

In [17]:
class ndarray(ndarray):
    def __add__(self, other):
        if isinstance(other, ndarray):
            if self.shape != other.shape:
                raise ValueError("Shapes must match.")

            return ndarray(
                self._apply_op(
                    self.data,
                    other.data,
                    lambda a, b: a + b
                )
            )

        else:
            raise NotImplementedError

Test it

In [18]:
# 1D
a = array([1,2,3])
b = array([4,5,6])

print(a + b)
# array([5,7,9])

# 2D
a = array([
    [1,2],
    [3,4]
])

b = array([
    [5,6],
    [7,8]
])

print(a + b)
# array([
#     [6,8],
#     [10,12]
# ])

array([5, 7, 9])
array([[6, 8], [10, 12]])


**Why recursion?**

Imagine we later support a 3D tensor.

```py
[
    [
        [1,2],
        [3,4]
    ],
    [
        [5,6],
        [7,8]
    ]
]
```

Our algorithm still works.

It keeps descending until it reaches numbers.

That's one of the beautiful ideas behind NumPy: operations are defined over arbitrary dimensions, not just vectors or matrices.

**Exercise:** Implement `__mul__`, `__sub__`, `__truediv__`

Our implementation still has a major limitation:

```py
a = array([1,2,3])

a + 10
```

doesn't work anymore because `_apply_op()` expects both operands to have the same nested structure.

The next feature we'll implement is scalar broadcasting. Once that's done, expressions like

```py
a + 10
a * 5

matrix + 100
matrix * 2
```

will all work naturally, and we'll be one step closer to real NumPy's broadcasting model.

But before that, let's give you an exercise to work on

# v7: Tuple Indexing
(Exercise)

In [19]:
a = np.array([
    [1,2,3],
    [4,5,6]
])

a[1,2]

np.int64(6)

But it doesn't work with our implementation

In [20]:
a = array([
    [1,2,3],
    [4,5,6]
])

a[1,2]

TypeError: list indices must be integers or slices, not tuple

# v8: Flat Array

At the moment, our ndarray only knows about 1D and 2D arrays. We've also stored the data as nested Python lists.

Real NumPy stores everything in a single **flat block of memory**, along with:

- shape
- strides (we will explain what this is, later)
- data type

Broadcasting is implemented by manipulating strides, not by creating new nested lists. That's what makes it both elegant and fast.

If we try to implement full broadcasting with our current nested-list representation, we'll end up writing a lot of special-case code for 1D vs 2D, 2D vs 3D, and so on. It works, but it doesn't teach the underlying design that NumPy uses.

First, let's create a helper function that turns a nester array into a flat array

In [3]:
def flatten(data):
    result = []
    if not isinstance(data, list):
        result.append(data)
    else:
        for item in data:
            result.extend(flatten(item))
    return result

flatten([
    [
        [1,2,3],
        [4,5,6]
    ],
    [
        [1,2,3],
        [4,5,6]
    ],
])

[1, 2, 3, 4, 5, 6, 1, 2, 3, 4, 5, 6]

Now we will store the flattened version of the data instead of the raw nested data

In [6]:
class ndarray:
    def __init__(self, data):
        self.data = flatten(data)

a = array([
    [
        [1,2,3],
        [4,5,6]
    ],
    [
        [1,2,3],
        [4,5,6]
    ],
])

a.data

[1, 2, 3, 4, 5, 6, 1, 2, 3, 4, 5, 6]

But we also need to keep track of the shape

In [7]:
def get_shape(data):
    if not isinstance(data, list):
        return ()
    return (len(data),) + get_shape(data[0])

In [8]:
class ndarray:
    def __init__(self, data):
        self.data = flatten(data)
        self.shape = get_shape(data)

a = array([
    [
        [1,2,3],
        [4,5,6]
    ],
    [
        [1,2,3],
        [4,5,6]
    ],
])

a.data, a.shape

([1, 2, 3, 4, 5, 6, 1, 2, 3, 4, 5, 6], (2, 2, 3))

Next, we validate that this issue we discussed earlier:
```py
[
    [1,2],
    [3,4,5]
]
```

In [9]:
def validate_shape(data, shape):

    if not isinstance(data, list):
        return

    if len(data) != shape[0]:
        raise ValueError("Irregular array.")

    for item in data:
        validate_shape(item, shape[1:])

In [10]:
class ndarray:
    def __init__(self, data):
        self.shape = get_shape(data)
        validate_shape(data, self.shape)
        self.data = flatten(data)

a = array([
    [
        [1,2],
        [4,5,6]
    ],
    [
        [1,2,3],
        [4,5,6]
    ],
])

a.data, a.shape

ValueError: Irregular array.

Now let's add the `__repr__` again

In [11]:
class ndarray(ndarray):
    def __repr__(self):
        return f"array({self.data}, shape={self.shape})"

a = array([
    [
        [1,2,3],
        [4,5,6]
    ],
    [
        [1,2,3],
        [4,5,6]
    ],
])

a

array([1, 2, 3, 4, 5, 6, 1, 2, 3, 4, 5, 6], shape=(2, 2, 3))

We will fix the `__repr__` later

For now, let's add indexing

How do you find an element in a flat array?

Suppose we have
```py
data = [1, 2, 3, 4, 5, 6]
shape = (2, 3)
```

This represents
```py
[
    [1, 2, 3],
    [4, 5, 6]
]
```

If I ask for
```py
a[1][2]
```
how do we know the answer is 6?

Think about the memory

Imagine the elements are stored consecutively.
```yml
Index:  0  1  2  3  4  5
Data : [1, 2, 3, 4, 5, 6]
```

The first row occupies indices 0–2.

The second row occupies indices 3–5.

So to go from row 0 to row 1, we skip 3 elements.

Where does the 3 come from?

It is simply the number of columns.

Another example

Suppose
```py
shape = (4, 5)
[
    [x x x x x]
    [x x x x x]
    [x x x x x]
    [x x x x x]
]
```

To move down one row, how many elements do we skip?

5.

Again, that's the number of columns.

What about 3D?

Suppose

shape = (2, 3, 4)

This means

2 blocks
each block has 3 rows
each row has 4 columns

To move

one column → skip 1
one row → skip 4
one block → skip 12 (3 × 4)

These numbers are called the strides.

shape   = (2, 3, 4)
strides = (12, 4, 1)
Can we compute the strides?

Yes.

Look at the pattern.
```
shape      stride
------     ------
4            1
3            4
2           12
```

Each stride is the product of all dimensions to its right.

Let's implement that.

In [12]:
def get_strides(shape):
    strides = []
    stride = 1

    for size in reversed(shape):
        strides.insert(0, stride)
        stride *= size

    return tuple(strides)

# Try it.

print(get_strides((3,)))
# (1,)

print(get_strides((2, 3)))
# (3, 1)

print(get_strides((2, 3, 4)))
# (12, 4, 1)


(1,)
(3, 1)
(12, 4, 1)


In [13]:
# Now modify your constructor.

class ndarray(ndarray):

    def __init__(self, data):
        self.shape = get_shape(data)
        validate_shape(data, self.shape)

        self.data = flatten(data)
        self.strides = get_strides(self.shape)

# Try it

a = array([
    [1,2,3],
    [4,5,6]
])

print(a.shape)
print(a.strides)

(2, 3)
(3, 1)


**Why are strides useful?**

Suppose someone asks for

row = 1
col = 2

Using the strides,

flat_index =
row * row_stride +
col * col_stride

which is

1 * 3 + 2 * 1 = 5

Then

data[5]

is

6

No nested lists.
No searching.
Just arithmetic.

In the next stage, we'll implement true multidimensional indexing:

a[1, 2]

instead of

a[1][2]

We'll write a function that converts an index tuple like (1, 2) into a flat index using the strides we just computed. This is exactly how NumPy accesses elements internally.

The goal is to make this work:
```py
a = array([
    [1, 2, 3],
    [4, 5, 6]
])

print(a[1, 2])
# 6
```

In [14]:
# Python passes (1, 2) as a tuple to __getitem__.
# Let's verify.

class Test:

    def __getitem__(self, index):
        print(index)


t = Test()

t[1,2]

(1, 2)


In [15]:
# Now, we convert a multidimensional index into a flat index

def get_flat_index(index, strides):
    flat = 0

    for i, stride in zip(index, strides):
        flat += i * stride

    return flat


print(get_flat_index((1, 2), (3, 1)))
print(get_flat_index((0, 1), (3, 1)))

5
1


In [16]:

class ndarray(ndarray):
    def __getitem__(self, index):
        if not isinstance(index, tuple):
            index = (index,)
        flat = get_flat_index(index, self.strides)

        return self.data[flat]

# Try it
a = array([
    [1,2,3],
    [4,5,6]
])

print(a[0,0])
print(a[0,2])
print(a[1,0])
print(a[1,2])

1
3
4
6


Next we'll implement `__setitem__`, so assignments like
```py
a[1, 2] = 100
```
work using the same indexing logic.

`__setitem__` is almost free now because we've already solved the hard part: converting multidimensional indices to flat indices.


In [17]:
class ndarray(ndarray):
    def __setitem__(self, index, value):

        if not isinstance(index, tuple):
            index = (index,)

        flat = get_flat_index(index, self.strides)

        self.data[flat] = value


# Try it
a = array([
    [1,2,3],
    [4,5,6]
])

a[0,1] = 20
print(a)

# Internally,
# data = [1,20,3,4,5,6]

a[1,2] = 100
print(a)

# Internally,
# data = [1,20,3,4,5,100]

array([1, 20, 3, 4, 5, 6], shape=(2, 3))
array([1, 20, 3, 4, 5, 100], shape=(2, 3))


Our current `__repr__` prints something like
```py
array([1, 20, 3, 4, 5, 100], shape=(2,3))
```

That's useful for debugging, but it doesn't look like NumPy.

We'd rather see
```py
array([
    [1, 20, 3],
    [4, 5, 100]
])
```

The problem is that we no longer store nested lists.

So we need to reconstruct them from
```
data = [1,20,3,4,5,100]
shape = (2,3)
```

Let's write a helper.

In [18]:
def reshape(data, shape):

    if len(shape) == 1:
        return data

    size = 1
    for x in shape[1:]:
        size *= x

    result = []

    for i in range(0, len(data), size):
        result.append(
            reshape(data[i:i+size], shape[1:])
        )

    return result

# Try it

data = [1,2,3,4,5,6]
print(reshape(data, (2,3)))

data = list(range(8))
print(reshape(data, (2,2,2)))

# Notice that this works for any number of dimensions.

[[1, 2, 3], [4, 5, 6]]
[[[0, 1], [2, 3]], [[4, 5], [6, 7]]]


In [19]:
# Update __repr__

class ndarray(ndarray):
    def __repr__(self):
        return f"array({reshape(self.data, self.shape)}, shape={self.shape})"

# Try it

a = array([
    [1,2,3],
    [4,5,6]
])

print(a)

# Not quite as pretty as NumPy's formatting, but much nicer than showing the flat buffer.

array([[1, 2, 3], [4, 5, 6]], shape=(2, 3))


# v9: Reshape

We're now going to build one of NumPy's coolest features.

> **Reshaping an array without moving any data.**

This is one of the reasons NumPy is so fast.

Suppose we have

```python
a = array([
    [1,2,3],
    [4,5,6]
])
```

Internally we have

```python
data = [1,2,3,4,5,6]
shape = (2,3)
```

Now suppose we do

```python
b = a.reshape(3,2)
```

What should happen?

```
[
    [1,2],
    [3,4],
    [5,6]
]
```

Did the data change?

No.

```
Before:

[1,2,3,4,5,6]

After:

[1,2,3,4,5,6]
```

Exactly the same.

Only the **interpretation** changes.

So, how do we implement it?

First things first: Count the elements

Before reshaping we must verify that both shapes contain the same number of elements.

Let's write a helper.

In [20]:
def numel(shape):
    total = 1

    for size in shape:
        total *= size

    return total

# Try it

numel((2,3))
numel((3,2))
numel((2,2))

4

Next, let's observe how numpy handles the `.reshape(...)` method

In [21]:
a = np.array([
    [1,2,3],
    [3,4,5]
])

a.reshape((3,2))
a.shape

(2, 3)

Strange, isn't it? The shape didn't change.

Now, let's try something else.

In [25]:
a = np.array([
    [1,2,3],
    [3,4,5]
])
b = a.reshape((3,2))
print(b.shape)
print(a.shape)

(3, 2)
(2, 3)


Okay, so `.reshape(...)` creates a new object, and returns it, instead of updating the shape of the vector itself.

But, if `b` is an entirely new object, changing the elements of `b` should not effect `a` at all.

Let's try

In [26]:
b[0,0] = 100
print(b)
print(a)

[[100   2]
 [  3   3]
 [  4   5]]
[[100   2   3]
 [  3   4   5]]


Interesting. The first element of both of them changed. As if, they are sharing the same internal `data`

In [ ]:
class ndarray(ndarray):
    def reshape(self, shape):
        if numel(self.shape) != numel(shape):
            raise ValueError("Total elements must be equal")
        
        new_obj = ndarray(self.data)
        new_obj.shape = shape
        return new_obj

a = array([
    [1,2,3],
    [2,3,4]
])

b = a.reshape((3,2))
print(b)
print(a)

array([[1, 2], [3, 2], [3, 4]], shape=(3, 2))
array([[1, 2, 3], [2, 3, 4]], shape=(2, 3))


So far so good. Let's do some more tests

In [28]:
b[0] = 100
print(a)
print(b)

array([[1, 2, 3], [2, 3, 4]], shape=(2, 3))
array([[100, 2], [3, 2], [3, 4]], shape=(3, 2))


Didn't work as expected. Somehow the internal data being used is not the same anymore. So, changing `b` does not change `a`.

We should think about what reshape really means.

It does not mean

> "Create another array."

It means

> "Look at the same bytes differently."

That's why NumPy treats reshape as creating a view, instead a creating another array.

Let's analyze why our code didn't work the way we expected.



Right now, `self.data` is still a nested list.



```

[

    [1,2,3],

    [2,3,4]

]

```



So



```python

ndarray(self.data)

```



doesn't copy anything—it simply stores the same nested list.



But after we switch to the flat representation,



```python

self.data = [1,2,3,2,3,4]

```



your constructor will likely do



```python

flatten(data)

```



which creates a brand new list.



So



```python

b = a.reshape((3,2))

```



would accidentally allocate new memory.

Right now, `self.data` is still a nested list.

```
[
    [1,2,3],
    [2,3,4]
]
```

So

```python
ndarray(self.data)
```

doesn't copy anything—it simply stores the same nested list.

But after we switch to the flat representation,
```python
self.data = [1,2,3,2,3,4]
```

our constructor will do this:
```python
flatten(data)
```

which creates a brand new list.

So
```python
b = a.reshape((3,2))
```
would accidentally allocate new memory.


So, what do we do now?

Our constructor currently expects nested lists.

But now we already have

```
data = [1,2,3,4,5,6]
```

We don't want to flatten it again.

So let's add an alternative constructor.

In [29]:
class ndarray(ndarray):
    @classmethod
    def from_buffer(cls, data, shape):

        obj = cls.__new__(cls)

        obj.data = data
        obj.shape = shape
        obj.strides = get_strides(shape)

        return obj


Notice something new:

```
cls.__new__(cls)
```

This creates an object **without calling `__init__()`**.

We don't want `__init__()` because it would flatten the data again.

Next, let's implement the new reshape method

In [30]:
class ndarray(ndarray):
    def reshape(self, *shape):

        if numel(shape) != numel(self.shape):
            raise ValueError("Cannot reshape array.")

        return ndarray.from_buffer(
            self.data,
            shape
        )


# Try it

a = array([
    [1,2,3],
    [4,5,6]
])

b = a.reshape(3,2)

print(a)
print(b)


array([[1, 2, 3], [4, 5, 6]], shape=(2, 3))
array([[1, 2], [3, 4], [5, 6]], shape=(3, 2))


In [31]:
# Did we copy the data?

b[0,0] = 100
a


array([[100, 2, 3], [4, 5, 6]], shape=(2, 3))

**Why is this fast?**

Imagine a huge array:

```
10000 × 10000
```

That's **100 million numbers**.

If reshaping copied all of them, it would take time and memory.

Instead, NumPy simply changes shape and strides, which takes essentially constant time.

Next: Transpose

Transpose is even more surprising.

You might think

```
[[1,2,3],
 [4,5,6]]
```

becomes

```
[[1,4],
 [2,5],
 [3,6]]
```

by moving data around.

In NumPy, **it usually doesn't**.

We'll derive how transposition can be implemented by changing only the **shape** and **strides**, leaving the data buffer untouched. That's one of the most elegant ideas in the entire library.

# v10: Transpose